In [ ]:
!pip install -q ultralytics albumentations ensemble-boxes

In [ ]:
import sys
import subprocess

# 1. 필수 라이브러리 체크 및 설치
try:
    import ultralytics
except ImportError:
    print("📦 ultralytics 및 필수 패키지 설치 중...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "albumentations", "ensemble-boxes"])

import os
import glob
import shutil
import random
import cv2
import torch
import numpy as np
import albumentations as A
from google.colab import drive
from ultralytics import YOLO

# 2. 구글 드라이브 마운트
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("✅ 구글 드라이브 마운트 확인 완료")

# 3. 경로 정의
DRIVE_DATA_DIR = "/content/drive/MyDrive/Pothole"
LOCAL_DATA_DIR = "/content/Pothole_Local"

# 4. 드라이브 -> 로컬 SSD 고속 복사
print("\n📦 [1/5] 구글 드라이브 데이터 -> 코랩 로컬 SSD 고속 복사 중...")
if os.path.exists(LOCAL_DATA_DIR):
    shutil.rmtree(LOCAL_DATA_DIR)

shutil.copytree(
    DRIVE_DATA_DIR,
    LOCAL_DATA_DIR,
    ignore=shutil.ignore_patterns('runs*', '*.pt', '*.cache', 'test_corrupted', '*.csv', '*.png')
)
print("✅ 로컬 복사 완료!")

# 5. 도로 환경 특화 증강 클래스
class PotholeDataAugmenter:
    def __init__(self):
        self.base_transform = A.Compose([
            A.MotionBlur(blur_limit=(7, 15), p=0.5),
            A.Defocus(radius=(3, 6), p=0.3),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
            A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=0.3)
        ])

    def apply_wet_asphalt_and_glint(self, image):
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        lab = cv2.merge((l, a, b))
        wet_img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

        h, w, _ = image.shape
        if random.random() < 0.6:
            glint_mask = np.zeros((h, w), dtype=np.float32)
            cx, cy = random.randint(int(w*0.2), int(w*0.8)), random.randint(int(h*0.3), int(h*0.9))
            rx, ry = random.randint(30, 100), random.randint(10, 40)
            angle = random.randint(0, 180)
            cv2.ellipse(glint_mask, (cx, cy), (rx, ry), angle, 0, 360, 1.0, -1)
            glint_mask = cv2.GaussianBlur(glint_mask, (77, 77), 0)
            glint_bgr = cv2.cvtColor((glint_mask * 180).astype(np.uint8), cv2.COLOR_GRAY2BGR)
            wet_img = cv2.addWeighted(wet_img, 1.0, glint_bgr, 0.4, 0)
        return wet_img

    def synthesize_shadows(self, image):
        h, w, _ = image.shape
        shadow_mask = np.ones((h, w), dtype=np.float32)
        num_polygons = random.randint(1, 3)
        for _ in range(num_polygons):
            pts = np.array([
                [random.randint(0, w), random.randint(0, int(h*0.5))],
                [random.randint(0, w), random.randint(0, int(h*0.5))],
                [random.randint(0, w), random.randint(int(h*0.5), h)],
                [random.randint(0, w), random.randint(int(h*0.5), h)]
            ], np.int32)
            poly_mask = np.zeros((h, w), dtype=np.uint8)
            cv2.fillPoly(poly_mask, [pts], 255)
            shadow_factor = random.uniform(0.3, 0.55)
            shadow_mask[poly_mask == 255] *= shadow_factor
        return (image.astype(np.float32) * shadow_mask[:, :, np.newaxis]).astype(np.uint8)

    def transform_pipeline(self, image):
        img = self.apply_wet_asphalt_and_glint(image)
        img = self.synthesize_shadows(img)
        return self.base_transform(image=img)['image']

# 6. YOLO 경로 인식을 위한 표준 디렉터리(train_aug/images, train_aug/labels) 생성 및 증강
print("\n🎨 [2/5] Train 세트 오프라인 데이터 증강 진행 중...")

ORIG_TRAIN_IMG_DIR = os.path.join(LOCAL_DATA_DIR, "train/images")
ORIG_TRAIN_LBL_DIR = os.path.join(LOCAL_DATA_DIR, "train/labels")

# 표준 YOLO 경로 규칙 준수 (train_aug/images <-> train_aug/labels)
AUG_TRAIN_IMG_DIR = os.path.join(LOCAL_DATA_DIR, "train_aug/images")
AUG_TRAIN_LBL_DIR = os.path.join(LOCAL_DATA_DIR, "train_aug/labels")

os.makedirs(AUG_TRAIN_IMG_DIR, exist_ok=True)
os.makedirs(AUG_TRAIN_LBL_DIR, exist_ok=True)

train_img_paths = glob.glob(os.path.join(ORIG_TRAIN_IMG_DIR, "*.*"))
print(f"  - 원본 학습 이미지 감지 수: {len(train_img_paths)}장")

augmenter = PotholeDataAugmenter()
generated_count = 0

for img_path in train_img_paths:
    filename = os.path.basename(img_path)
    base_name = os.path.splitext(filename)[0]
    lbl_path = os.path.join(ORIG_TRAIN_LBL_DIR, base_name + ".txt")

    # 원본 복사
    shutil.copy2(img_path, os.path.join(AUG_TRAIN_IMG_DIR, filename))
    if os.path.exists(lbl_path):
        shutil.copy2(lbl_path, os.path.join(AUG_TRAIN_LBL_DIR, base_name + ".txt"))
    generated_count += 1

    # 증강 2장 생성
    img = cv2.imread(img_path)
    if img is None:
        continue

    for i in range(2):
        aug_img = augmenter.transform_pipeline(img)
        aug_filename = f"{base_name}_envaug_{i}.jpg"
        cv2.imwrite(os.path.join(AUG_TRAIN_IMG_DIR, aug_filename), aug_img)

        if os.path.exists(lbl_path):
            shutil.copy2(lbl_path, os.path.join(AUG_TRAIN_LBL_DIR, f"{base_name}_envaug_{i}.txt"))
        generated_count += 1

print(f"✅ 오프라인 증강 완료! 총 학습 이미지/라벨 세트: {generated_count}개")

# 7. data.yaml 자동 작성 (train 경로를 train_aug/images로 지정)
print("\n📝 [3/5] data.yaml 경로 설정 작성 중...")
yaml_path = os.path.join(LOCAL_DATA_DIR, "data.yaml")
yaml_content = f"""
path: {LOCAL_DATA_DIR}
train: train_aug/images
val: valid/images
test: test/images

nc: 1
names: ['pothole']
""".strip()

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)
print("✅ data.yaml 작성 완료")

# 8. YOLOv8 n / s / m 모델군 순차 학습
print("\n🚀 [4/5] YOLOv8 Multi-Scale (n/s/m) 순차 학습 개시...")
device_setting = 0 if torch.cuda.is_available() else "cpu"
SAVE_PROJECT_DIR = os.path.join(DRIVE_DATA_DIR, "runs")

models_to_train = [
    ("yolov8n", "yolov8n.pt", 16),
    ("yolov8s", "yolov8s.pt", 16),
    ("yolov8m", "yolov8m.pt", 8)
]

for model_name, weights, batch_size in models_to_train:
    print(f"\n==================================================")
    print(f"🔥 [{model_name}_pothole_env_aug] 학습 시작 (장치: {device_setting}, Batch: {batch_size})")
    print(f"==================================================")

    model = YOLO(weights)
    model.train(
        data=yaml_path,
        epochs=150,
        imgsz=1024,
        batch=batch_size,
        optimizer="AdamW",
        lr0=0.001,
        patience=30,
        project=SAVE_PROJECT_DIR,
        name=f"{model_name}_pothole_env_aug",
        exist_ok=True,
        device=device_setting
    )

print("\n🏆 [5/5] 모든 파이프라인 학습 프로세스 완료!")
print(f"📁 최종 가중치 저장 위치: {SAVE_PROJECT_DIR}")

In [ ]:
import os
import glob
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion

# 1. 학습된 3개 모델 로드
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Pothole/runs"
model_n = YOLO(os.path.join(DRIVE_PROJECT_DIR, "yolov8n_pothole_env_aug/weights/best.pt"))
model_s = YOLO(os.path.join(DRIVE_PROJECT_DIR, "yolov8s_pothole_env_aug/weights/best.pt"))
model_m = YOLO(os.path.join(DRIVE_PROJECT_DIR, "yolov8m_pothole_env_aug/weights/best.pt"))

models = [model_n, model_s, model_m]
weights = [1.0, 1.0, 1.5]  # 가장 성능이 좋았던 m 모델에 높은 가중치 부여

def predict_wbf(img_path, iou_thr=0.5, skip_box_thr=0.25):
    img = cv2.imread(img_path)
    h, w, _ = img.shape

    boxes_list = []
    scores_list = []
    labels_list = []

    for model in models:
        results = model.predict(img_path, imgsz=1024, verbose=False)[0]
        if len(results.boxes) == 0:
            boxes_list.append([])
            scores_list.append([])
            labels_list.append([])
            continue

        # [x1, y1, x2, y2] 정규화 (0 ~ 1)
        boxes = results.boxes.xyxyn.cpu().numpy().tolist()
        scores = results.boxes.conf.cpu().numpy().tolist()
        labels = results.boxes.cls.cpu().numpy().astype(int).tolist()

        boxes_list.append(boxes)
        scores_list.append(scores)
        labels_list.append(labels)

    # WBF 실행
    fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        boxes_list, scores_list, labels_list,
        weights=weights, iou_thr=iou_thr, skip_box_thr=skip_box_thr
    )

    # 픽셀 좌표 복원
    fused_boxes_pixel = []
    for box in fused_boxes:
        x1, y1, x2, y2 = box
        fused_boxes_pixel.append([int(x1 * w), int(y1 * h), int(x2 * w), int(y2 * h)])

    return fused_boxes_pixel, fused_scores, fused_labels

In [ ]:
# 1. 테스트 세트에 대한 단일 모델 최적 평가 (TTA 적용)
best_model_path = os.path.join(DRIVE_PROJECT_DIR, "yolov8m_pothole_env_aug/weights/best.pt")
model = YOLO(best_model_path)

print("🧪 Test 데이터셋 최종 평가 진행 중 (TTA 적용)...")
test_results = model.val(
    data="/content/Pothole_Local/data.yaml",
    split="test",       # test 분할 세트 평가
    imgsz=1024,
    augment=True,       # Test-Time Augmentation (TTA) 활성화
    project=DRIVE_PROJECT_DIR,
    name="test_evaluation_results"
)

print(f"📊 Test mAP50: {test_results.box.map50:.4f}")
print(f"📊 Test mAP50-95: {test_results.box.map:.4f}")